# OOP Week 3 -- Cleaner Component

**Course:** Object-Oriented Programming (Year 2)
**Session:** 3 hours
**Prerequisites:** Weeks 1-2 (Classes, Composition)
**Focus:** strategy selection, composing cleaners, method overriding

---

## Learning Objectives

1. Build a base cleaner class with a template method
2. Create specialized cleaner subclasses
3. Compose multiple cleaners into a pipeline
4. Understand method overriding (polymorphism preview)
5. Track cleaning metadata (drop counts, reasons)

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/ArifSolmaz/courseos-curriculum.git
# %cd courseos-curriculum/course-content

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Setup: Dataset Class (from Week 2)

In [ ]:
class Dataset:
    """Data container from Week 2."""
    def __init__(self, rows, source_path="unknown"):
        self.rows = rows
        self.source_path = source_path
        self.n_rows = len(rows)
        self.columns = list(rows[0].keys()) if rows else []

    def get_column(self, name):
        return [row.get(name) for row in self.rows]

    def __len__(self):
        return self.n_rows

    def __str__(self):
        return "Dataset(" + str(self.n_rows) + " rows)"


# Sample data for this session
sample_data = Dataset([
    {"id": 1, "value": 25.0, "status": "ok"},
    {"id": 2, "value": 30.0, "status": "ok"},
    {"id": 3, "value": 88.0, "status": "warning"},
    {"id": 4, "value": -5.0, "status": "error"},
    {"id": 5, "value": None, "status": "ok"},
    {"id": 6, "value": 200.0, "status": "ok"},
    {"id": 7, "value": 42.0, "status": ""},
], "sensors.csv")
print("Sample:", sample_data)

**Expected Output:**
```
Sample: Dataset(7 rows)
```

---
## Section 1: The Base Cleaner Pattern

Instead of writing one giant cleaning function, we define a **base class** that provides the cleaning loop, and **subclasses** that define specific cleaning rules.

This is called the **Template Method** pattern: the base class defines the algorithm skeleton, and subclasses fill in the details.

In [ ]:
class BaseCleaner:
    """Base cleaner with template method pattern.

    Subclasses override check() to define their cleaning rule.
    The clean() method handles the loop and tracking.
    """

    def __init__(self, name="base"):
        self.name = name
        self.drop_reasons = {}
        self.n_checked = 0
        self.n_dropped = 0

    def clean(self, dataset):
        """Clean a dataset. Returns a NEW Dataset."""
        clean_rows = []
        self.drop_reasons = {}
        self.n_checked = 0
        self.n_dropped = 0

        for row in dataset.rows:
            self.n_checked += 1
            reason = self.check(row)
            if reason is None:
                clean_rows.append(row)
            else:
                self.n_dropped += 1
                self.drop_reasons[reason] = self.drop_reasons.get(reason, 0) + 1

        return Dataset(clean_rows, dataset.source_path)

    def check(self, row):
        """Check one row. Return reason string if invalid, None if ok.

        Override this in subclasses!
        """
        return None  # base class accepts everything

    def report(self):
        """Print a cleaning report."""
        print("Cleaner: " + self.name)
        print("  Checked: " + str(self.n_checked))
        print("  Dropped: " + str(self.n_dropped))
        for reason, count in self.drop_reasons.items():
            print("    - " + reason + ": " + str(count))


# The base cleaner accepts everything
base = BaseCleaner()
result = base.clean(sample_data)
print("Base cleaner kept:", len(result), "of", len(sample_data))
base.report()

**Expected Output:**
```
Base cleaner kept: 7 of 7
Cleaner: base
  Checked: 7
  Dropped: 0
```

---
## Section 2: Specialized Cleaners

Now we create cleaners that override `check()` with specific rules.

### RangeCleaner -- drops values outside a range

In [ ]:
class RangeCleaner(BaseCleaner):
    """Drop rows where a numeric column is outside a range."""

    def __init__(self, column, low, high):
        super().__init__("range_" + column)
        self.column = column
        self.low = low
        self.high = high

    def check(self, row):
        val = row.get(self.column)
        if isinstance(val, (int, float)):
            if val < self.low or val > self.high:
                return "out_of_range_" + self.column
        return None


rc = RangeCleaner("value", 0, 100)
result = rc.clean(sample_data)
rc.report()
print("Kept values:", result.get_column("value"))

**Expected Output:**
```
Cleaner: range_value
  Checked: 7
  Dropped: 2
    - out_of_range_value: 2
Kept values: [25.0, 30.0, 88.0, None, 42.0]
```

### MissingCleaner -- drops rows with None or empty strings

In [ ]:
class MissingCleaner(BaseCleaner):
    """Drop rows with missing values in specified columns."""

    def __init__(self, columns):
        super().__init__("missing")
        self.columns = columns

    def check(self, row):
        for col in self.columns:
            val = row.get(col)
            if val is None:
                return "missing_" + col
            if isinstance(val, str) and val.strip() == "":
                return "empty_" + col
        return None


mc = MissingCleaner(["value", "status"])
result = mc.clean(sample_data)
mc.report()
print("Kept:", len(result), "rows")

**Expected Output:**
```
Cleaner: missing
  Checked: 7
  Dropped: 2
    - missing_value: 1
    - empty_status: 1
Kept: 5 rows
```

---
## Section 3: Composing Cleaners

The power of this design is that you can **chain** multiple cleaners. Each one applies its own rule. The output of one becomes the input of the next.

In [ ]:
class CleaningPipeline:
    """Applies multiple cleaners in sequence."""

    def __init__(self, cleaners=None):
        self.cleaners = cleaners or []

    def add(self, cleaner):
        self.cleaners.append(cleaner)
        return self  # allows chaining: pipeline.add(a).add(b)

    def clean(self, dataset):
        """Apply all cleaners in sequence."""
        current = dataset
        for cleaner in self.cleaners:
            current = cleaner.clean(current)
        return current

    def report(self):
        """Print reports for all cleaners."""
        total_dropped = 0
        for c in self.cleaners:
            c.report()
            total_dropped += c.n_dropped
        print("Total dropped: " + str(total_dropped))


# Build the pipeline
pipeline = CleaningPipeline()
pipeline.add(MissingCleaner(["value", "status"]))
pipeline.add(RangeCleaner("value", 0, 100))

# Run it
print("Input:", sample_data)
clean = pipeline.clean(sample_data)
print("Output:", clean)
print("Clean values:", clean.get_column("value"))
print()
pipeline.report()

**Expected Output:**
```
Input: Dataset(7 rows)
Output: Dataset(3 rows)
Clean values: [25.0, 30.0, 88.0]

Cleaner: missing
  Checked: 7
  Dropped: 2
    - missing_value: 1
    - empty_status: 1
Cleaner: range_value
  Checked: 5
  Dropped: 2
    - out_of_range_value: 2
Total dropped: 4
```

---
### Cleaner Class Hierarchy

```
    +------------------+
    |   BaseCleaner    |
    +------------------+
    | - name           |
    | - drop_reasons   |
    +------------------+
    | + clean(dataset) |
    | + check(row)     |  <-- override this
    | + report()       |
    +------------------+
         /          \\
        /            \\
+---------------+  +------------------+
| RangeCleaner  |  | MissingCleaner   |
+---------------+  +------------------+
| - column      |  | - columns        |
| - low, high   |  +------------------+
+---------------+  | + check(row)     |
| + check(row)  |  +------------------+
+---------------+
```

---
### Try It!

Create a `StatusCleaner(BaseCleaner)` that drops rows where `status` is `"error"`. Then add it to the pipeline and run again.

In [ ]:
# YOUR CODE HERE


---
### Common Mistake: Forgetting super().__init__() in subclass

The code below has a bug. Can you spot it before reading the fix?

In [ ]:
class BrokenCleaner(BaseCleaner):
    def __init__(self, threshold):
        # BUG: forgot super().__init__()!
        self.threshold = threshold

bc = BrokenCleaner(50)
try:
    bc.clean(sample_data)
except AttributeError as e:
    print("ERROR:", e)

**What goes wrong:** `super().__init__()` calls the parent class constructor. Without it, the BaseCleaner attributes (`name`, `drop_reasons`, etc.) are never created, causing AttributeError when `clean()` tries to use them.

**The fix:**

In [ ]:
class FixedCleaner(BaseCleaner):
    def __init__(self, threshold):
        super().__init__("threshold")  # FIXED!
        self.threshold = threshold

fc = FixedCleaner(50)
result = fc.clean(sample_data)
print("Works! Kept", len(result), "rows")

---
## Section 4: More Cleaner Types

### TypeCleaner -- drops rows with wrong types

In [ ]:
class TypeCleaner(BaseCleaner):
    """Drop rows where a column is not the expected type."""

    def __init__(self, column, expected_type):
        super().__init__("type_" + column)
        self.column = column
        self.expected_type = expected_type

    def check(self, row):
        val = row.get(self.column)
        if not isinstance(val, self.expected_type):
            return "wrong_type_" + self.column
        return None


tc = TypeCleaner("value", (int, float))
result = tc.clean(sample_data)
tc.report()
print("Kept values:", result.get_column("value"))

**Expected Output:**
```
Cleaner: type_value
  Checked: 7
  Dropped: 1
    - wrong_type_value: 1
Kept values: [25.0, 30.0, 88.0, -5.0, 200.0, 42.0]
```

### OutlierCleaner -- drops statistical outliers

In [ ]:
class OutlierCleaner(BaseCleaner):
    """Drop rows where a value is more than N std devs from mean."""

    def __init__(self, column, n_std=2):
        super().__init__("outlier_" + column)
        self.column = column
        self.n_std = n_std
        self._mean = 0
        self._std = 0

    def clean(self, dataset):
        # Pre-compute stats before checking rows
        vals = [r.get(self.column) for r in dataset.rows
                if isinstance(r.get(self.column), (int, float))]
        if vals:
            self._mean = sum(vals) / len(vals)
            self._std = (sum((x - self._mean)**2 for x in vals) / len(vals)) ** 0.5
        return super().clean(dataset)

    def check(self, row):
        val = row.get(self.column)
        if isinstance(val, (int, float)) and self._std > 0:
            z_score = abs(val - self._mean) / self._std
            if z_score > self.n_std:
                return "outlier_" + self.column
        return None


data_with_outlier = Dataset([
    {"id": 1, "value": 10.0},
    {"id": 2, "value": 12.0},
    {"id": 3, "value": 11.0},
    {"id": 4, "value": 13.0},
    {"id": 5, "value": 100.0},  # outlier!
], "test.csv")

oc = OutlierCleaner("value", n_std=2)
result = oc.clean(data_with_outlier)
oc.report()
print("Kept:", result.get_column("value"))

**Expected Output:**
```
Cleaner: outlier_value
  Checked: 5
  Dropped: 1
    - outlier_value: 1
Kept: [10.0, 12.0, 11.0, 13.0]
```

---
## Section 5: Cleaning Pipeline with Full Report

In [ ]:
# Build a comprehensive pipeline
full_pipeline = CleaningPipeline()
full_pipeline.add(TypeCleaner("value", (int, float)))
full_pipeline.add(MissingCleaner(["status"]))
full_pipeline.add(RangeCleaner("value", 0, 100))

print("Input:", sample_data, "-- values:", sample_data.get_column("value"))
print()

clean = full_pipeline.clean(sample_data)
print()
print("Output:", clean, "-- values:", clean.get_column("value"))
print()
full_pipeline.report()

**Expected Output:**
```
Input: Dataset(7 rows) -- values: [25.0, 30.0, 88.0, -5.0, None, 200.0, 42.0]

Output: Dataset(3 rows) -- values: [25.0, 30.0, 88.0]

Cleaner: type_value
  Checked: 7
  Dropped: 1
Cleaner: missing
  Checked: 6
  Dropped: 1
Cleaner: range_value
  Checked: 5
  Dropped: 2
Total dropped: 4
```

---
### Procedural vs OOP: Data Cleaning

The procedural version packs all rules into one function. Adding a new rule means modifying that function (violating OCP). The OOP version lets you add/remove/reorder rules without touching any existing code.

**Procedural approach (what you did in CP1/CP2):**

In [ ]:
# PROCEDURAL: one big function with many if-statements
def clean_all(data, min_v, max_v, required_cols):
    result = []
    for row in data:
        # Check type
        if not isinstance(row.get('value'), (int, float)):
            continue
        # Check missing
        skip = False
        for col in required_cols:
            if row.get(col) is None or row.get(col) == '':
                skip = True
                break
        if skip:
            continue
        # Check range
        if row['value'] < min_v or row['value'] > max_v:
            continue
        result.append(row)
    return result

# Hard to add new rules without modifying this function!

**OOP approach (what we are learning now):**

In [ ]:
# OOP: composable, each rule is independent
pipeline = CleaningPipeline()
pipeline.add(TypeCleaner('value', (int, float)))
pipeline.add(MissingCleaner(['status']))
pipeline.add(RangeCleaner('value', 0, 100))
# Easy to add new rules: pipeline.add(OutlierCleaner('value'))

clean = pipeline.clean(dataset)
pipeline.report()  # detailed per-step reporting!

---
### Debugging Tip: Cleaners running in wrong order

The order of cleaners matters! If you put RangeCleaner before TypeCleaner, the RangeCleaner might crash on non-numeric values.

**Best practice:** Always put type/missing cleaners FIRST, then range/logic cleaners SECOND.

```
Recommended order:
1. TypeCleaner (remove wrong types)
2. MissingCleaner (remove nulls/empties)
3. RangeCleaner (remove out-of-range)
4. OutlierCleaner (remove statistical outliers)
```

---
### Try It!

Create a `StatusCleaner(BaseCleaner)` that drops rows where the `status` column contains `"error"`. Test it with the sample data, then add it to the full pipeline.

In [ ]:
# YOUR CODE HERE


---
### Try It!

Create a `WhitespaceCleaner(BaseCleaner)` that drops rows where a specified column is all whitespace. Test with:
```python
data = Dataset([
    {"name": "Alice", "note": "good"},
    {"name": "Bob", "note": "   "},  # whitespace only
    {"name": "Charlie", "note": ""},  # empty
])
```

In [ ]:
# YOUR CODE HERE


---
## Build from Scratch Exercise

This exercise tests whether you truly understand this week's concepts. Complete it without looking at the examples above.

In [ ]:
# BUILD FROM SCRATCH:
# Build a cleaning pipeline with 3 custom cleaners for student grade data: RemoveFailing (< 0), RemoveOutliers (> 100), RemoveIncomplete (None grades). Use BaseCleaner pattern.

# YOUR CODE HERE


In [ ]:
# TEST your build-from-scratch code:

# YOUR TESTS HERE


---
## Connect the Dots

How does this week's concept connect to previous weeks?

In [ ]:
# How does the Cleaner use composition (Week 2) and classes (Week 1)?

# YOUR ANSWER (as comments or code):


---
## Real-World Spotting

OOP patterns are everywhere in real software. Can you spot them?

In [ ]:
# Think about a mail sorting system. What 'cleaning' rules would you apply? How would you compose them?

# YOUR ANSWER:


---
## Diagram It

Draw an ASCII class diagram for the main classes from this week. Include:
- Class names
- Key attributes
- Key methods
- Relationships (has-a, is-a)

In [ ]:
# Draw your ASCII diagram here:
# +------------------+
# |   ClassName      |
# +------------------+
# | - attribute      |
# +------------------+
# | + method()       |
# +------------------+

# YOUR DIAGRAM:


---
## Key Vocabulary

| Term | Definition |
|------|------------|
| **Template Method** | Base class defines algorithm skeleton, subclasses fill in details |
| **Override** | Subclass provides its own version of a parent method |
| **`super()`** | Call the parent class version of a method |
| **Pipeline** | A sequence of processing steps |
| **Drop reason** | Why a row was removed during cleaning |

---
## Recap Exercise

Without looking at the code above, try to:

In [ ]:
# 1. Write one class from this week FROM MEMORY
#    (it does not need to be perfect)

# YOUR CODE HERE


# 2. Create an instance and call at least one method

# YOUR CODE HERE


# 3. Write one test for your class

# YOUR CODE HERE


---
## What to Review Before Next Week

Before the next session, make sure you can:

1. Explain this week's main concept in your own words
2. Write a simple example from memory
3. Identify this pattern in existing code
4. Explain WHY this pattern is useful (not just HOW)

---
## Mini-Quiz

In [ ]:
# Q1: What does the BaseCleaner.check() method return for valid rows?
# Answer: 

# Q2: Why do we use super().__init__() in subclass constructors?
# Answer: 

# Q3: What is the Template Method pattern?
# Answer: 

# Q4: What advantage does a CleaningPipeline have over one big cleaner?
# Answer: 

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)